In [ ]:
import requestsimport reimport pandas as pdfrom pathlib import Pathfrom datetime import datetime

In [ ]:
HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; AtlasFinanceBot/1.0)"}API_URL = "https://data.gov.ma/data/api/3/action/package_search"params = {"fq": "organization:ammc", "q": "OPCVM hebdo", "rows": 20}resp = requests.get(API_URL, params=params, headers=HEADERS, timeout=30)data = resp.json()data["result"]["count"]

In [ ]:
packages = data["result"]["results"][p["name"] for p in packages]

In [ ]:
def extract_year(name):    m = re.search(r"(20\d{2})", name)    return int(m.group(1)) if m else 0target_pkg = max(packages, key=lambda p: extract_year(p["name"]))target_pkg["name"]

In [ ]:
def extract_resource_date(resource):    m = re.search(r"(\d{2})[-_]?(\d{2})[-_]?(\d{4})", resource["url"])    if not m:        return None    d, mo, y = m.groups()    return datetime(int(y), int(mo), int(d))resources_with_dates = [(r, extract_resource_date(r)) for r in target_pkg["resources"]]resources_with_dates = [rd for rd in resources_with_dates if rd[1] is not None]resources_with_dates.sort(key=lambda rd: rd[1], reverse=True)latest_resource, latest_date = resources_with_dates[0]latest_date, latest_resource["url"]

In [ ]:
raw_folder = Path("../data/raw/ammc")raw_folder.mkdir(parents=True, exist_ok=True)filename = latest_resource["url"].split("/")[-1]filepath = raw_folder / filenamer = requests.get(latest_resource["url"], headers=HEADERS, timeout=60)with open(filepath, "wb") as f:    f.write(r.content)filepath

In [ ]:
week_date = latest_dateraw = pd.read_excel(filepath, sheet_name=0, header=None)raw.shape

In [ ]:
mask = raw[1].astype(str).str.contains("Actif net par cat", case=False, na=False)start_actif_net = raw[mask].index[0]mask = raw[1].astype(str).str.contains("Indices De Performance", case=False, na=False)start_indices = raw[mask].index[0]mask = raw[1].astype(str).str.contains("Actif Total Par Cat", case=False, na=False)start_actif_total = raw[mask].index[0]mask = raw[1].astype(str).str.contains("Souscriptions et rachats", case=False, na=False)start_flux = raw[mask].index[0]start_actif_net, start_indices, start_actif_total, start_flux

In [ ]:
df_actif_net = raw.iloc[start_actif_net+4:start_actif_net+11, 1:8].copy()df_actif_net.columns = ["categorie", "nombre_opcvm", "montant_mad", "structure_pct", "variation_hebdo_pct", "variation_mensuelle_pct", "variation_annuelle_pct"]df_actif_net["semaine_du"] = week_datedf_actif_net

In [ ]:
df_indices = raw.iloc[start_indices+4:start_indices+9, 1:6].copy()df_indices.columns = ["categorie", "indice", "variation_hebdo_pct", "variation_mensuelle_pct", "variation_annuelle_pct"]df_indices["semaine_du"] = week_datedf_indices

In [ ]:
df_actif_total = raw.iloc[start_actif_total+4:start_actif_total+17, 1:7].copy()df_actif_total.columns = ["categorie", "montant_mad", "structure_pct", "variation_hebdo_pct", "variation_mensuelle_pct", "variation_annuelle_pct"]df_actif_total["semaine_du"] = week_datedf_actif_total

In [ ]:
df_flux = raw.iloc[start_flux+2:start_flux+4, 1:9].copy()df_flux.columns = raw.iloc[start_flux+1, 1:9].tolist()df_flux["semaine_du"] = week_datedf_flux

In [ ]:
df_actif_net.to_csv("../data/raw/ammc_actif_net_extracted.csv", index=False)df_indices.to_csv("../data/raw/ammc_indices_extracted.csv", index=False)df_actif_total.to_csv("../data/raw/ammc_actif_total_extracted.csv", index=False)df_flux.to_csv("../data/raw/ammc_flux_extracted.csv", index=False)